# 02 — Radiometric Preprocessing (Illumination Normalization)

**Input:** `26166_DATA/processed/geotiff/{instrument}/*.tif` from Notebook 01
**Output:** `26166_DATA/processed/radiometric/{instrument}/*.tif`


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, csv
import numpy as np
import rasterio
import cv2
import matplotlib.pyplot as plt

DRIVE_ROOT = '/content/drive/MyDrive/26166_DATA'
IN_ROOT    = f'{DRIVE_ROOT}/processed/geotiff'
OUT_ROOT   = f'{DRIVE_ROOT}/processed/radiometric'
os.makedirs(OUT_ROOT, exist_ok=True)

with open(f'{DRIVE_ROOT}/processed/manifest.csv') as f:
    manifest_rows = list(csv.DictReader(f))
print(f'{len(manifest_rows)} products in manifest')


## 1. Build a matchable single-band grayscale for every product

In [ ]:
def to_uint8(band, low_pct=1, high_pct=99):
    band = band.astype(np.float32)
    valid = band[np.isfinite(band)]
    lo, hi = np.percentile(valid, [low_pct, high_pct])
    band = np.clip((band - lo) / max(hi - lo, 1e-6), 0, 1)
    return (band * 255).astype(np.uint8)

def iirs_to_grayscale(cube, method='pca'):
    bands, h, w = cube.shape
    if method == 'mean':
        return cube.mean(axis=0)
    flat = cube.reshape(bands, -1).astype(np.float32).T
    flat -= flat.mean(axis=0, keepdims=True)
    sample_idx = np.random.choice(flat.shape[0], size=min(200_000, flat.shape[0]), replace=False)
    _, _, vt = np.linalg.svd(flat[sample_idx], full_matrices=False)
    pc1 = flat @ vt[0]
    return pc1.reshape(h, w)


## 2. Illumination-invariance: CLAHE (+ optional homomorphic filtering)

In [ ]:
def apply_clahe(img_u8, clip_limit=2.0, tile_size=8):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
    return clahe.apply(img_u8)

def homomorphic_filter(img_u8, gamma_low=0.5, gamma_high=1.5, cutoff=30):
    img = img_u8.astype(np.float32) + 1.0
    log_img = np.log(img)
    rows, cols = img.shape
    crow, ccol = rows // 2, cols // 2
    u, v = np.meshgrid(np.arange(cols) - ccol, np.arange(rows) - crow)
    d = np.sqrt(u**2 + v**2)
    h = (gamma_high - gamma_low) * (1 - np.exp(-(d**2) / (2 * (cutoff**2)))) + gamma_low
    fft = np.fft.fftshift(np.fft.fft2(log_img))
    img_back = np.fft.ifft2(np.fft.ifftshift(fft * h)).real
    return to_uint8(np.exp(img_back) - 1.0)


## 3. Run over every converted product

In [ ]:
processed_rows = []

for row in manifest_rows:
    instrument = row['instrument']
    with rasterio.open(row['geotiff_path']) as src:
        arr = src.read()
        profile = src.profile.copy()

    gray_u8 = to_uint8(iirs_to_grayscale(arr, 'pca')) if (instrument == 'iirs' and arr.shape[0] > 1) \
              else to_uint8(arr[0])
    gray_clahe = apply_clahe(gray_u8)

    out_dir = f'{OUT_ROOT}/{instrument}'
    os.makedirs(out_dir, exist_ok=True)
    out_path = f"{out_dir}/{row['product']}.tif"

    profile.update({'count': 1, 'dtype': 'uint8'})
    profile.pop('crs', None)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(gray_clahe[np.newaxis, ...])

    processed_rows.append({**row, 'radiometric_path': out_path})

print(f'Processed {len(processed_rows)} products.')


## 4. Before / after visual check

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for col, instrument in enumerate(['ohrc', 'tmc2', 'iirs']):
    rows = [r for r in processed_rows if r['instrument'] == instrument]
    if not rows:
        continue
    r = rows[0]
    with rasterio.open(r['geotiff_path']) as src:
        raw = src.read()
    raw_gray = iirs_to_grayscale(raw, 'pca') if instrument == 'iirs' and raw.shape[0] > 1 else raw[0]
    raw_u8 = to_uint8(raw_gray)
    with rasterio.open(r['radiometric_path']) as src:
        processed = src.read(1)
    axes[0, col].imshow(raw_u8, cmap='gray'); axes[0, col].set_title(f'{instrument} — before'); axes[0, col].axis('off')
    axes[1, col].imshow(processed, cmap='gray'); axes[1, col].set_title(f'{instrument} — after CLAHE'); axes[1, col].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
out_manifest = f'{DRIVE_ROOT}/processed/manifest_radiometric.csv'
with open(out_manifest, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=processed_rows[0].keys())
    writer.writeheader()
    writer.writerows(processed_rows)
print('Saved', out_manifest)


---
Next notebook: **03_geometric_preprocess.ipynb**